In [1]:
from json import dump, load
from os import listdir
from pandas import DataFrame, read_excel, concat
from tqdm import tqdm
from chembl_webresource_client.new_client import new_client
from time import sleep
import requests as rq


In [ ]:
new_data = {}
files = listdir('../toxval')
files.remove('dumps')
for file in tqdm(files):
    data = read_excel(f'../toxval/{file}')
    data = data[data['TOXVAL_TYPE'] == 'LD50']
    if not data.empty:
        data = concat((
            data['DTXSID'],
            data['TOXVAL_NUMERIC'],
            data['TOXVAL_UNITS'],
            data['SPECIES_COMMON'],
            data['EXPOSURE_ROUTE']
        ), axis=1)
        
        for i, name in enumerate(data['DTXSID']):
            info = [
                {
                "animal": data.iloc[i, 3],
                "injection": data.iloc[i, 4],
                "value": data.iloc[i, 1],
                "measure": data.iloc[i, 2]
            }]
            new_data[name] = new_data.get(name, []) + info
            print(len(new_data))

In [7]:
with open('new_data_DTXSID.json', 'w', encoding='utf-8') as file:
    dump(new_data, file, indent=4)

In [9]:
print(len(new_data))

36343


In [ ]:
new_data_smi = {}
for name, value in tqdm(new_data.items()):
    flag = True
    while flag:
        try:
            smi = rq.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/smiles/txt', timeout=5)
            if smi.status_code == 200:
                new_data_smi[smi.text.rstrip('\n')] = new_data_smi.get(smi.text.rstrip('\n'), []) + value
            else:
                new_data_smi[name+'ERROR'] = value 
            flag = False
        except rq.exceptions.Timeout:
            print(name)
            continue


In [ ]:
with open('new_data_smi_v2.json', 'w', encoding='utf-8') as file:
    dump(new_data_smi, file, indent=4)

In [17]:
len(new_data_smi)

36310

In [18]:
sum(len(el) for el in new_data_smi.values())

66115

In [24]:
with open('new_data_smi_v1.json', 'r', encoding='utf-8') as file:
    data = load(file)
    print(sum(len(el) for el in data.values()))

54255
